# RoBERTa detector (role B)
No logic here: every cell just calls `src/train_roberta.py`. Settings: GPU T4, Internet on, dataset `team20-m4-processed` attached.
Run order follows issue #10: `indist` first (keep its model), then LOGO, then paraphrase, then cross-domain. Commit/download results as you go; stop if GPU quota runs out.

In [ ]:
!git clone -b main https://github.com/NIRMAL20062/generated_text_detection.git
%cd generated_text_detection
# If pip breaks torch, comment this line out: Kaggle's preinstalled packages are enough.
!pip install -q -r requirements.txt
# The dataset mounts under /kaggle/input/... (path differs by account), so find it.
!mkdir -p data && ln -sfn "$(dirname "$(find /kaggle/input -name m4_en.parquet | head -1)")" data/processed
!ls -l data/processed/   # must show m4_en.parquet
!python -m src.data --check-splits

## 1. `indist`: train, then score the original paraphrase test set with the same model

In [ ]:
!python -m src.train_roberta --setting indist

In [ ]:
!python -m src.train_roberta --setting paraphrase_orig --predict-only --model outputs/roberta/indist/model

## 2. Leave-one-generator-out (most important for the research question)
`--no-save` keeps disk free: only the `indist` model is needed later.

In [ ]:
for gen in ["chatGPT", "davinci", "cohere", "dolly", "bloomz"]:
    !python -m src.train_roberta --setting logo_{gen} --no-save

## 3. Paraphrase attack (only after Vishal's issue #13 lands `splits/paraphrase*.json` and `paraphrased.parquet`)

In [ ]:
import os
for s in ["paraphrase", "paraphrase_human"]:
    if os.path.exists(f"splits/{s}.json"):
        !python -m src.train_roberta --setting {s} --predict-only --model outputs/roberta/indist/model
    else:
        print(f"splits/{s}.json not on this branch yet: skip, rerun this cell after #13 merges")

## 4. Cross-domain

In [ ]:
for dom in ["wikipedia", "wikihow", "reddit", "arxiv", "peerread"]:
    !python -m src.train_roberta --setting xdomain_{dom} --no-save

## 5. Metrics with the shared evaluator

In [ ]:
# src/eval.py is only on the a-data branch until PR #19 merges, so fetch just that file if it is missing.
import glob, os, subprocess, sys, urllib.request
EVAL_URL = "https://raw.githubusercontent.com/NIRMAL20062/generated_text_detection/a-data/src/eval.py"
if not os.path.exists("src/eval.py"):
    urllib.request.urlretrieve(EVAL_URL, "src/eval.py")
for p in sorted(glob.glob("results/roberta/*/preds.csv")):
    r = subprocess.run([sys.executable, "-m", "src.eval", p], capture_output=True, text=True)
    print((r.stdout.strip().splitlines() or [""])[0] if r.returncode == 0 else f"FAILED {p}: {r.stderr.strip().splitlines()[-1]}")

In [ ]:
!ls results/roberta   # download this folder from the Output tab, then commit it from your own machine